# 01 · ETL de la encuesta de estrés percibido y preferencias turísticas

**Fuente:** Google Forms, 9 oct 2026, 139 respuestas (archivo crudo en `data/encuesta/raw/`, fuera de git).
**Salida:** `data/encuesta/encuesta_limpia.csv` (anónimo, sin marca temporal) y su bitácora `encuesta_limpia.log.json`.

El código del ETL vive en `src/encuesta_etl.py`; este notebook lo ejecuta y muestra cada decisión.

| Paso | Regla |
|---|---|
| Extraer | CSV de Google Forms, columnas renombradas por posición |
| Excluir | menores de 18 años; respuestas duplicadas (idénticas sin marca temporal); sin respuesta en P10 |
| Unificar | variantes ortográficas ("Precoupación" → "Preocupación") |
| Codificar | escalas ordinales a números; selección múltiple (P7, P8, P10) a columnas 0/1; ocupación agrupada |
| Índices | **estrés 0-100** (P3, P4 y P5a-d), **tensión 0-100** (P5), **bienestar 0-100** (P6); P5/P6 toleran 1 ítem faltante |
| Anonimizar | ID `R001…`; se elimina la marca temporal |

In [1]:
import sys
from pathlib import Path
SRC = next(p for p in [Path('src'), Path('../src'), Path('MODELO/src')] if (p / 'encuesta_etl.py').exists()).resolve()
sys.path.insert(0, str(SRC))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lab_graficas import aplicar_estilo, guardar, SERIES, TEXTO_2, ETIQUETAS_PREF, ETIQUETAS_W
aplicar_estilo()
pd.set_option('display.max_columns', 50)

In [2]:
from encuesta_etl import ejecutar, cargar_crudo, RAW_DEFAULT, OUT_DEFAULT
crudo = cargar_crudo()
print('Crudo:', crudo.shape)
limpio, bitacora = ejecutar()
pd.Series({k: v for k, v in bitacora.items() if not isinstance(v, dict)}).to_frame('valor')

Crudo: (139, 18)


,valor
filas_crudas,139
excluidas_menores,1
excluidas_duplicadas,1
excluidas_sin_p10,0
indice_estres_vacio,11
indice_bienestar_vacio,10
filas_limpias,137
sha256_crudo,27c3830c9523005d257a017c3ec30acb4b32f1e44d908b...


## Respuestas abiertas que no corresponden a una opción
Se dejan fuera de las columnas 0/1 ("Ninguna de las anteriores" equivale a no marcar opciones; "Orar" aparece una sola vez).

In [3]:
bitacora['respuestas_abiertas_no_mapeadas']

{'p7_manifestaciones': ['Ninguna de las anteriores'],
 'p8_estresores': ['No identifico situaciones que me generen estrés'],
 'p10_experiencias': ['Orar']}

## Datos faltantes por variable (después de limpiar)

In [4]:
faltantes = limpio.isna().sum()
faltantes[faltantes > 0].to_frame('faltantes')

,faltantes
p5a_dificultad_relajarse,9
p5b_nerviosismo,5
p5c_irritabilidad,9
p5d_concentracion,10
p6a_buen_animo,9
p6b_tranquilo,6
p6c_energia,8
p6d_descansado,10
usaria_app,1
estres_0_100,11


In [5]:
limpio.head()

,id_respuesta,edad_rango,edad_ord,ocupacion,p3,p4,p5a_dificultad_relajarse,p5b_nerviosismo,p5c_irritabilidad,p5d_concentracion,p6a_buen_animo,p6b_tranquilo,p6c_energia,p6d_descansado,satisfaccion,usaria_app,estres_0_100,tension_0_100,bienestar_0_100,sint_cansancio,sint_tension_muscular,sint_sueno,sint_sobrepasado,sint_preocupacion,sint_decisiones,estr_trabajo_estudios,estr_falta_tiempo,estr_familia,estr_economia,estr_incertidumbre,estr_traslados,estr_relaciones,pref_cascadas_rios,pref_caminatas,pref_relajacion,pref_hospedaje_tranquilo,pref_cultural,pref_gastronomia,pref_baja_intensidad,pref_comunidades
0,R001,De 18 a 24 años,1,otro,3,3,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2,1.0,47.2,33.3,33.3,1,1,1,1,1,1,1,1,1,1,1,0,0,1,0,0,1,1,1,1,0
1,R002,De 18 a 24 años,1,otro,3,3,2.0,2.0,2.0,2.0,2.0,2.0,1.0,0.0,2,1.0,69.4,66.7,41.7,1,1,1,1,1,0,1,1,1,1,1,1,0,0,1,1,1,0,1,0,0
2,R003,De 18 a 24 años,1,estudia_y_trabaja,4,1,0.0,2.0,2.0,2.0,3.0,2.0,3.0,1.0,2,1.0,54.2,50.0,75.0,1,1,0,1,0,0,1,1,0,1,1,0,1,1,1,1,1,1,1,1,1
3,R004,De 18 a 24 años,1,estudiante,3,2,1.0,2.0,2.0,2.0,3.0,2.0,3.0,2.0,1,1.0,59.7,58.3,83.3,1,1,1,1,1,0,1,0,1,1,1,0,1,0,1,1,1,0,1,0,0
4,R005,De 18 a 24 años,1,estudiante,3,3,2.0,2.0,2.0,2.0,2.0,1.0,0.0,1.0,0,1.0,69.4,66.7,33.3,1,1,1,1,0,1,1,0,1,1,0,0,1,0,0,1,1,0,0,1,0


## Diccionario de variables

In [6]:
dicc = pd.DataFrame({
    'variable': ['edad_ord', 'p3, p4', 'p5a-p5d', 'p6a-p6d', 'satisfaccion', 'estres_0_100', 'tension_0_100',
                 'bienestar_0_100', 'sint_*', 'estr_*', 'pref_*'],
    'significado': ['0=<18 … 5=55+', 'Frecuencia último mes, 0 Nunca - 4 Muy frecuentemente',
                    'Última semana, 0 Nunca - 3 Casi siempre', 'Últimas 2 semanas, 0 Nunca - 3 Casi todo el tiempo',
                    'P9, 0 Nada - 4 Completamente', '% del máximo de P3, P4, P5 (mayor = más estrés)',
                    '% del máximo de P5', '% del máximo de P6 (mayor = más bienestar)',
                    'P7 manifestaciones de estrés (0/1)', 'P8 situaciones estresantes (0/1)',
                    'P10 experiencias que ayudarían a descansar (0/1) — variable objetivo'],
})
dicc

,variable,significado
0,edad_ord,0=<18 … 5=55+
1,"p3, p4","Frecuencia último mes, 0 Nunca - 4 Muy frecuen..."
2,p5a-p5d,"Última semana, 0 Nunca - 3 Casi siempre"
3,p6a-p6d,"Últimas 2 semanas, 0 Nunca - 3 Casi todo el ti..."
4,satisfaccion,"P9, 0 Nada - 4 Completamente"
5,estres_0_100,"% del máximo de P3, P4, P5 (mayor = más estrés)"
6,tension_0_100,% del máximo de P5
7,bienestar_0_100,% del máximo de P6 (mayor = más bienestar)
8,sint_*,P7 manifestaciones de estrés (0/1)
9,estr_*,P8 situaciones estresantes (0/1)
